[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/01.%20The%20Single%20Crane%20Lift%20Sequence%20Problem/P1-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 1. The Single Crane Lift Sequence Problem

## Tier 3 — Genetic Algorithm for Near-Optimal Sequencing

This notebook uses a genetic algorithm (GA), a population-based stochastic search method, to find a good container-lifting sequence. A GA can explore many candidate sequences, but it does not certify that its best-found sequence is globally optimal.

The implementation is written from scratch so that the representation, evaluation, selection, crossover, mutation, and stopping rule are visible. The GA searches over container orderings; a deterministic simulation converts each ordering into single and dual lifts and evaluates its objective.

### Learning goals

- Understand the GA loop: initialize, evaluate, select, recombine, mutate, and repeat.
- Interpret a chromosome as a container permutation and connect it to the crane plan produced by the simulator.
- Read a best-so-far convergence curve and distinguish improvement from a proof of optimality.
- Separate simulated operating time from the priority penalty included in the search objective.

### Inputs and outputs

The input is a fixed instance of eight containers, their grid locations, weights, priorities, and crane-time parameters. The output is the best ordering found during the run, its decoded lift plan, its objective value, and visual summaries of the plan.

### Scope and modeling assumptions

The storage layout is a 2-by-4 grid. Crane travel uses Manhattan distance, and a dual lift is allowed only for two edge-adjacent containers whose combined weight does not exceed the weight limit. The crane starts at the specified coordinate in single-lift mode. The decoder considers consecutive containers in the chromosome from left to right: it takes a feasible dual lift whenever the current container and the next one form a feasible pair; otherwise, it takes a single lift. The model is deterministic and omits uncertainty, blocking, release times, and other operational constraints.

In [ ]:
# Check the required environment without installing packages.
# A managed course environment should provide these dependencies in advance.
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib in your environment."
    ) from e

# A fixed seed makes a run reproducible when the notebook is executed from the top.
rng = np.random.default_rng(42)
print("Dependencies imported successfully.")


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
from itertools import combinations


# Each container has an identifier, a grid coordinate (row, column),
# a weight, and a priority level.
@dataclass(frozen=True)
class Container:
    id: int
    rc: Tuple[int, int]
    weight: float
    priority: int


# Fixed eight-container instance.
containers = [
    Container(1, (0, 0), 8.5, 3),
    Container(2, (0, 1), 12.3, 2),
    Container(3, (0, 2), 15.7, 4),
    Container(4, (0, 3), 9.8, 3),
    Container(5, (1, 0), 11.2, 2),
    Container(6, (1, 1), 14.1, 2),
    Container(7, (1, 2), 13.9, 1),
    Container(8, (1, 3), 10.6, 2),
]

id_to_container = {container.id: container for container in containers}


# Time parameters are measured in minutes.
HANDLE_SINGLE = 2.5
HANDLE_DUAL = 3.8
SETUP_TIME = 1.2
DUAL_WEIGHT_LIMIT = 20.0
TRAVEL_PER_GRID_STEP = 0.8

START_RC = (0.0, 0.0)
START_MODE = "single"


def manhattan(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    """Return grid travel distance using horizontal and vertical moves."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def adjacent_int(a: Tuple[int, int], b: Tuple[int, int]) -> bool:
    """Return whether two grid cells share an edge."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1


def dual_feasible(i: int, j: int) -> bool:
    """Check the adjacency and combined-weight requirements for a dual lift."""
    ci, cj = id_to_container[i], id_to_container[j]
    return (
        adjacent_int(ci.rc, cj.rc)
        and ci.weight + cj.weight <= DUAL_WEIGHT_LIMIT
    )


def center(ci: Container, cj: Container) -> Tuple[float, float]:
    """Return the coordinate midway between two container locations."""
    return (
        (ci.rc[0] + cj.rc[0]) / 2.0,
        (ci.rc[1] + cj.rc[1]) / 2.0,
    )


def evaluate_order(order: List[int]) -> Tuple[float, List[Dict[str, Any]]]:
    """Decode a container permutation and return its objective and lift plan.

    The objective is simulated operating time plus the priority penalty.
    Each plan step records its operating duration and its share of that penalty
    separately, so the physical timeline does not treat the penalty as elapsed time.
    """
    expected_ids = set(id_to_container)
    if len(order) != len(expected_ids) or set(order) != expected_ids:
        raise ValueError("order must contain each container ID exactly once")

    total_operating_time = 0.0
    rc = START_RC
    mode = START_MODE
    plan: List[Dict[str, Any]] = []

    # The chromosome gives a sequence; this decoder groups each feasible
    # consecutive pair into one dual lift and otherwise performs a single lift.
    i = 0
    while i < len(order):
        cid = order[i]

        if i + 1 < len(order) and dual_feasible(order[i], order[i + 1]):
            a, b = order[i], order[i + 1]
            ca, cb = id_to_container[a], id_to_container[b]
            target = center(ca, cb)

            travel = TRAVEL_PER_GRID_STEP * manhattan(rc, target)
            setup = SETUP_TIME if mode != "dual" else 0.0
            step_time = travel + setup + HANDLE_DUAL

            total_operating_time += step_time
            plan.append(
                {
                    "type": "dual",
                    "containers": [a, b],
                    "to": target,
                    "travel_time": travel,
                    "setup_time": setup,
                    "handling_time": HANDLE_DUAL,
                    "step_time": step_time,
                }
            )
            rc = target
            mode = "dual"
            i += 2

        else:
            container = id_to_container[cid]
            target = (float(container.rc[0]), float(container.rc[1]))

            travel = TRAVEL_PER_GRID_STEP * manhattan(rc, target)
            setup = SETUP_TIME if mode != "single" else 0.0
            step_time = travel + setup + HANDLE_SINGLE

            total_operating_time += step_time
            plan.append(
                {
                    "type": "single",
                    "containers": [cid],
                    "to": target,
                    "travel_time": travel,
                    "setup_time": setup,
                    "handling_time": HANDLE_SINGLE,
                    "step_time": step_time,
                }
            )
            rc = target
            mode = "single"
            i += 1

    # The priority penalty is a soft objective term, not a hard constraint.
    # It is assigned to the lift step that handles each container.
    priority_penalty = 0.0
    denominator = max(1, len(plan) - 1)
    for k, step in enumerate(plan):
        step_penalty = sum(
            0.05
            * (4 - id_to_container[cid].priority)
            * (k / denominator)
            for cid in step["containers"]
        )
        step["priority_penalty"] = step_penalty
        step["objective_cost"] = step["step_time"] + step_penalty
        priority_penalty += step_penalty

    objective = total_operating_time + priority_penalty
    assert np.isclose(
        objective,
        sum(step["objective_cost"] for step in plan),
    )
    return float(objective), plan


# Evaluate a simple ordering to demonstrate the objective function interface.
v0, _ = evaluate_order(list(range(1, 9)))
v0


## Genetic algorithm

### Objective and chromosome representation

A chromosome is a permutation of the container IDs. The decoder in `evaluate_order` turns that permutation into a lift plan. It always uses a feasible dual lift when the next two IDs are adjacent in the ordering and satisfy the adjacency and weight conditions; all other current containers are lifted singly.

For a decoded plan with lift steps indexed by $k$, the search objective is

$$J = \sum_k t_k + \sum_k p_k,$$

where $t_k$ is the operating duration of step $k$, including travel, any mode-change setup, and handling, and $p_k$ is the soft priority penalty assigned to that step. For a step that handles container set $C_k$, the penalty is

$$p_k = \sum_{i \in C_k} 0.05(4 - q_i)\frac{k}{\max(1, K-1)}.$$

Here, $q_i$ is container $i$'s priority and $K$ is the number of lift steps. The step index begins at zero, so earlier lifts receive a smaller position factor; a higher priority value also receives a smaller penalty. This term influences which chromosome the GA prefers, but it is not a hard deadline or precedence constraint. It is a search score, not elapsed operating time. Accordingly, the returned plan stores `step_time` and `priority_penalty` separately, and its physical completion time is the sum of `step_time` values.

### Search operators and loop

- **Tournament selection** samples candidate chromosomes and chooses the one with the lowest objective. This favors good candidates while retaining variation in the population.
- **Order crossover (OX)** copies a slice from one parent and fills the remaining positions in the order in which unused IDs appear in the other parent. The child remains a permutation: no container is lost or duplicated.
- **Swap mutation** exchanges two positions with a specified probability. This introduces new orderings into the population.
- **Elitism** copies the best candidates into the next generation unchanged, so the best solution found so far cannot be lost from the population.

The algorithm initializes a population of random permutations, evaluates each one, and repeats selection, crossover, mutation, and replacement for a fixed number of generations. The recorded history is the best objective found up to each generation; it is therefore non-increasing. A flat section means that the run has not found a better candidate during those generations, not that optimality has been proved.

With population size $P$, $G$ generations, and $n$ containers, decoding and evaluating one chromosome takes $O(n)$ time. The order-crossover implementation uses membership checks in a list and takes $O(n^2)$ time in the worst case. Thus, a broad upper bound for this implementation is $O(GP n^2)$, with additional tournament evaluations of the same order. The GA's solution quality is not guaranteed by this runtime bound.

In [ ]:
def tournament_select(pop: List[List[int]], k: int = 3) -> List[int]:
    """Select the lowest-objective chromosome from a random tournament."""
    if not 1 <= k <= len(pop):
        raise ValueError("tournament size k must be between 1 and population size")

    indices = rng.choice(len(pop), size=k, replace=False)
    best = None
    best_value = float("inf")

    for index in indices:
        value, _ = evaluate_order(pop[index])
        if value < best_value:
            best_value = value
            best = pop[index]

    assert best is not None
    return best


def order_crossover(p1: List[int], p2: List[int]) -> List[int]:
    """Combine two parent permutations using order crossover (OX)."""
    if len(p1) != len(p2) or set(p1) != set(p2):
        raise ValueError("parents must be permutations of the same container IDs")

    n = len(p1)
    if n < 2:
        return p1.copy()

    # Two distinct cut points ensure that the copied slice contains at least one gene.
    a, b = sorted(rng.choice(n, size=2, replace=False))
    child = [-1] * n
    child[a:b] = p1[a:b]

    # Preserve the order of the second parent while skipping IDs already copied.
    fill = [gene for gene in p2 if gene not in child]
    pointer = 0
    for index in range(n):
        if child[index] == -1:
            child[index] = fill[pointer]
            pointer += 1

    assert set(child) == set(p1)
    return child


def mutate_swap(order: List[int], rate: float = 0.15) -> List[int]:
    """With probability rate, swap two positions in a chromosome."""
    if not 0.0 <= rate <= 1.0:
        raise ValueError("mutation rate must be between 0 and 1")

    if len(order) >= 2 and rng.random() < rate:
        i, j = rng.choice(len(order), size=2, replace=False)
        order = order.copy()
        order[i], order[j] = order[j], order[i]

    return order


def run_ga(
    pop_size: int = 40,
    generations: int = 120,
    elite: int = 4,
    crossover_rate: float = 0.8,
    mutation_rate: float = 0.15,
):
    """Run the GA and return the best ordering, objective, plan, and history."""
    if pop_size < 2:
        raise ValueError("population size must be at least 2")
    if generations < 1:
        raise ValueError("generations must be at least 1")
    if not 1 <= elite < pop_size:
        raise ValueError("elite count must be at least 1 and smaller than population size")
    if not 0.0 <= crossover_rate <= 1.0:
        raise ValueError("crossover rate must be between 0 and 1")
    if not 0.0 <= mutation_rate <= 1.0:
        raise ValueError("mutation rate must be between 0 and 1")

    base = [container.id for container in containers]
    pop = [rng.permutation(base).tolist() for _ in range(pop_size)]

    best_value = float("inf")
    best_order = None
    history = []

    for _ in range(generations):
        scored = [(evaluate_order(individual)[0], individual) for individual in pop]
        scored.sort(key=lambda item: item[0])

        if scored[0][0] < best_value:
            best_value = scored[0][0]
            best_order = scored[0][1].copy()

        history.append(best_value)

        # Elitism preserves the strongest candidates without altering their genes.
        new_pop = [individual.copy() for _, individual in scored[:elite]]

        while len(new_pop) < pop_size:
            parent1 = tournament_select(pop)
            parent2 = tournament_select(pop)

            if rng.random() < crossover_rate:
                child1 = order_crossover(parent1, parent2)
                child2 = order_crossover(parent2, parent1)
            else:
                child1, child2 = parent1.copy(), parent2.copy()

            child1 = mutate_swap(child1, rate=mutation_rate)
            child2 = mutate_swap(child2, rate=mutation_rate)
            new_pop.extend([child1, child2])

        pop = new_pop[:pop_size]

    assert best_order is not None
    best_objective, best_plan = evaluate_order(best_order)
    assert np.isclose(best_objective, best_value)

    return best_order, best_objective, best_plan, history


# Execute the search using the fixed GA parameters and random seed above.
best_order, best_objective, best_plan, history = run_ga()

print("Best order found:", best_order)
print(f"Best objective score: {best_objective:.3f} min-equivalent")
print(
    "Simulated operating time: "
    f"{sum(step['step_time'] for step in best_plan):.3f} min"
)
print(
    "Priority penalty: "
    f"{sum(step['priority_penalty'] for step in best_plan):.3f} min-equivalent"
)

plan_table = pd.DataFrame(best_plan)
assert np.isclose(
    best_objective,
    plan_table["objective_cost"].sum(),
)
plan_table


In [ ]:
# The history records the best objective found up to each generation.
plt.figure(figsize=(7, 3))
plt.plot(history, lw=2, label="Best objective found")
plt.title("GA convergence: best-so-far objective")
plt.xlabel("Generation")
plt.ylabel("Objective score (min-equivalent)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Interpreting the best plan

The path visualization represents the crane targets in the plan returned by the GA. A single-lift target is the container's grid coordinate; a dual-lift target is the midpoint of the two container coordinates. Arrows connect successive targets, beginning at the crane's start coordinate.

Short moves between targets can indicate that the ordering groups nearby work. The path shows target locations and movement order; it does not display travel time, handling time, setup time, or the priority penalty separately.

In [ ]:
# Draw the storage grid, container labels, and the crane's target sequence.
fig, ax = plt.subplots(figsize=(7, 3.2))

for row in [0, 1]:
    for col in [0, 1, 2, 3]:
        ax.scatter(
            [col],
            [-row],
            s=900,
            marker="s",
            facecolor="#F2F4F7",
            edgecolor="#344054",
        )

for container in containers:
    x, y = container.rc[1], -container.rc[0]
    ax.text(
        x,
        y,
        f"{container.id}\n(w={container.weight}, p={container.priority})",
        ha="center",
        va="center",
        fontsize=9,
    )

path = [(START_RC[1], -START_RC[0])]
for step in best_plan:
    row, col = step["to"]
    path.append((col, -row))

for (x1, y1), (x2, y2) in zip(path[:-1], path[1:]):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", lw=2, color="#12B76A"),
    )

ax.set_title("GA best-found crane path")
ax.set_xlabel("Column coordinate")
ax.set_ylabel("Row coordinate (displayed downward)")
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(-0.75, 3.75)
ax.set_ylim(-1.75, 0.75)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## Operational visualizations

The following views derive from the decoded best plan. They show physical lift-step durations, cumulative operating time, and a timeline. These charts use `step_time`, which includes travel, setup, and handling but excludes the soft priority penalty. Therefore, the final cumulative operating time is not necessarily equal to the GA objective score.

In [ ]:
vis = pd.DataFrame(best_plan).copy()
vis.index = np.arange(1, len(vis) + 1)
vis["step"] = vis.index.astype(int)

# The timeline uses elapsed operating time, not the GA's priority-adjusted score.
vis["cum_time"] = vis["step_time"].cumsum()
vis["start_time"] = vis["cum_time"] - vis["step_time"]

color_map = {"single": "#2E90FA", "dual": "#12B76A"}
colors = [color_map[step_type] for step_type in vis["type"]]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))

axes[0].bar(
    vis["step"],
    vis["step_time"],
    color=colors,
    edgecolor="#101828",
    alpha=0.85,
)
axes[0].set_title("Operating duration by lift step")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Operating time (min)")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].plot(
    vis["step"],
    vis["cum_time"],
    marker="o",
    lw=2,
    color="#344054",
)
axes[1].set_title("Cumulative operating time")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Operating time (min)")
axes[1].grid(True, alpha=0.25)

axes[2].barh(
    y=vis["step"],
    width=vis["step_time"],
    left=vis["start_time"],
    color=colors,
    edgecolor="#101828",
    alpha=0.85,
)
axes[2].set_title("Operating timeline")
axes[2].set_xlabel("Operating time (min)")
axes[2].set_ylabel("Step")
axes[2].grid(True, axis="x", alpha=0.25)

from matplotlib.patches import Patch

legend_handles = [
    Patch(facecolor=color_map["single"], edgecolor="#101828", label="Single lift"),
    Patch(facecolor=color_map["dual"], edgecolor="#101828", label="Dual lift"),
]
axes[2].legend(handles=legend_handles, loc="best")

plt.tight_layout()
plt.show()


## Strengths and limitations of this search method

A GA is useful when an instance has many possible orderings and a practical search budget is available. Population size and generation count provide direct controls over the amount of search, while the objective function can incorporate additional modeled costs.

The method is stochastic and does not guarantee a globally optimal sequence. A single run can stop without finding the best possible ordering, and its outcome can depend on the random seed and GA parameters. The convergence plot reports progress within the run; it is not an optimality certificate. Increasing the population or number of generations can increase computational effort, but does not by itself prove that the result is optimal.

The quality of the result also depends on the decoder and model. Here, the decoder makes a fixed local decision: whenever the next two containers form a feasible dual pair, it lifts them together. The GA searches over orderings under that decoding rule; it does not independently choose among all possible lift plans for a given ordering.

## Controlled example: evaluating a feasible dual lift

A feature can be present in the simulator without appearing in the best-found plan. This controlled comparison uses the same instance and parameters to illustrate how the decoder represents a dual lift. It evaluates the baseline ordering and an ordering that places the first feasible pair consecutively at the start.

This comparison is between two specified orderings, not a general claim that dual lifts improve or worsen the objective. Their effect depends on travel, handling, setup, the rest of the sequence, and the priority penalty.

In [ ]:
# Identify all pairs that satisfy the model's dual-lift requirements.
feasible_pairs = []
for i, j in combinations([container.id for container in containers], 2):
    if dual_feasible(i, j):
        feasible_pairs.append((i, j))

print("Feasible dual pairs:", feasible_pairs)

baseline_order = [container.id for container in containers]
base_objective, base_plan = evaluate_order(baseline_order)

if not feasible_pairs:
    forced_order = baseline_order.copy()
    forced_objective, forced_plan = base_objective, base_plan
    print("No feasible dual pair exists under the current constraints.")
else:
    first, second = feasible_pairs[0]
    rest = [
        container_id
        for container_id in baseline_order
        if container_id not in (first, second)
    ]
    forced_order = [first, second] + rest
    forced_objective, forced_plan = evaluate_order(forced_order)

print("\nBaseline order:", baseline_order)
print(f"Baseline objective score: {base_objective:.3f} min-equivalent")
print("Plan step types:", [step["type"] for step in base_plan])

print("\nOrder with a consecutive feasible pair:", forced_order)
print(f"Objective score: {forced_objective:.3f} min-equivalent")
print("Plan step types:", [step["type"] for step in forced_plan])

comparison = pd.DataFrame(
    {
        "ordering": ["baseline", "consecutive_feasible_pair"],
        "objective_score": [base_objective, forced_objective],
        "operating_time": [
            sum(step["step_time"] for step in base_plan),
            sum(step["step_time"] for step in forced_plan),
        ],
        "priority_penalty": [
            sum(step["priority_penalty"] for step in base_plan),
            sum(step["priority_penalty"] for step in forced_plan),
        ],
        "num_dual_steps": [
            sum(step["type"] == "dual" for step in base_plan),
            sum(step["type"] == "dual" for step in forced_plan),
        ],
    }
)
comparison
